# DocVQA: task-relevant answer-region ablation

리뷰어 1의 코멘트 5 보완 실험입니다. SROIE 실행이 끝난 뒤 이 노트북을 별도로 실행하세요.

입력: `DocVQA-PII-val199-execution-data.zip`을 Google Drive의 `내 드라이브` 최상위에 넣습니다. 이미 있다면 다시 업로드할 필요가 없습니다.

전체 실행: `런타임 → 모두 실행`. GPU는 필수가 아닙니다. 모델은 Node.js에서 실행합니다.

194개 문서에 대해 기존 PII 삽입 이미지와 정답 영역을 검게 가린 이미지를 새로 평가합니다. 총 388회 추론입니다. 기존 실험 결과는 섞지 않습니다. 자동 정답 매칭 점수 1.0이 있는 문서만 포함하고, 같은 질문의 모든 기록된 정답 영역을 가립니다. 기록되지 않은 반복 정답은 남아 있을 수 있습니다. 좌표는 수동 검증된 주석이 아닙니다.

ANLS는 소문자와 공백을 정규화하고 구두점을 유지합니다. 정규화 편집 거리 0.5 이상은 0점입니다. 기존 분석 코드의 구두점 제거 방식과 다르므로 이전 ANLS 수치와 직접 비교하지 않습니다.

문서별 대응 차이, 10,000회 부트스트랩 신뢰구간과 원천 문서별 군집 부트스트랩을 산출합니다. 문서 역할은 실험용 역할 라벨이며 실제 문서 유형으로 해석하지 않습니다. 질문 유형별 결과도 별도로 산출합니다. 하위집단은 탐색적 분석입니다.

이 실험은 작업 정보 제거에 대한 모델의 민감도를 평가합니다. Risk-adaptive의 우월성, PII 유형별 효과 또는 실제 재식별 방지를 입증하는 실험은 아닙니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib, zipfile, platform, urllib.request

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive')
INPUT_ZIP = BASE / 'DocVQA-PII-val199-execution-data.zip'
if not INPUT_ZIP.exists():
    candidates = [BASE/'DocVQA_Current'/INPUT_ZIP.name, BASE/'DocVQA_TaskAblation'/INPUT_ZIP.name]
    INPUT_ZIP = next((p for p in candidates if p.exists()), INPUT_ZIP)
if not INPUT_ZIP.exists():
    raise FileNotFoundError('DocVQA-PII-val199-execution-data.zip을 내 드라이브 최상위에 업로드한 후 다시 실행하세요.')
WORK = Path('/content/docvqa_answer_ablation'); WORK.mkdir(exist_ok=True)
OUT = BASE/'DocVQA_TaskAblation_Results'; OUT.mkdir(exist_ok=True)
CACHE = BASE/'DocVQA_PII_Model_Cache'; CACHE.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','-q','install','pillow','numpy'],check=True)
subprocess.run(['npm','install','--silent','@xenova/transformers@2.17.2'],cwd=WORK,check=True)

def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
print('입력 확인:',INPUT_ZIP)
with zipfile.ZipFile(INPUT_ZIP) as z:
    for name in z.namelist():
        if name.startswith('images/') or name in ['docvqa_pii_val199.json','docvqa_qas_val199.json','qa_manifest.csv']:
            target=(WORK/name).resolve()
            if not target.is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP path')
            z.extract(name,WORK)


In [ ]:
SCRIPTS = {'prepare_ablation.py': 'import csv, json, hashlib, sys\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nfrom PIL import Image\n\nroot, out = map(Path, sys.argv[1:3]); out.mkdir(parents=True, exist_ok=True)\ndocs = {d[\'document_id\']: d for d in json.loads((root/\'docvqa_pii_val199.json\').read_text())[\'documents\']}\nqas = {str(q[\'questionId\']): q for q in json.loads((root/\'docvqa_qas_val199.json\').read_text())[\'data\']}\nrows = list(csv.DictReader((root/\'qa_manifest.csv\').open()))\nmanifest, audit = [], []\nfor r in rows:\n    d = docs[r[\'document_id\']]\n    matches = [m for m in d[\'qa_answer_matches\'] if str(m[\'question_id\']) == r[\'question_id\'] and m[\'localized\'] and m[\'box\']]\n    exact = any(m[\'match_score\'] == 1.0 for m in matches)\n    record = {\'document_id\':r[\'document_id\'], \'question_id\':int(r[\'question_id\']), \'role\':r[\'role\'], \'eligible\':exact,\'source_document_id\':d[\'source_ucsf_document_id\']}\n    if not exact:\n        record[\'exclusion_reason\'] = \'no_exact_localization_match\'; audit.append(record); continue\n    src = root/\'images\'/Path(r[\'image_file\']).name\n    image = Image.open(src).convert(\'RGB\'); a = np.array(image); h,w = a.shape[:2]\n    boxes = sorted(set(tuple(map(int,m[\'box\'])) for m in matches))\n    mask = np.zeros((h,w), dtype=bool)\n    for x1,y1,x2,y2 in boxes:\n        if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):\n            raise ValueError(f\'invalid box: {r["document_id"]} {(x1,y1,x2,y2)}\')\n        mask[y1:y2,x1:x2] = True\n    # Exclude synthetic-PII overlap to isolate removal of source answer information.\n    overlap = 0\n    for entity in d[\'entities\']:\n        x1,y1,x2,y2 = map(int,entity[\'box\']); overlap += int(mask[max(0,y1):min(h,y2),max(0,x1):min(w,x2)].sum())\n    if overlap:\n        record.update(eligible=False,exclusion_reason=\'synthetic_pii_overlap\',overlap_pixels=overlap); audit.append(record); continue\n    b = a.copy(); b[mask] = 0\n    assert np.array_equal(a[~mask], b[~mask]) and np.all(b[mask] == 0)\n    dest = out/\'answer_masked\'/src.name; dest.parent.mkdir(exist_ok=True)\n    Image.fromarray(b).save(dest)\n    record.update(boxes=boxes,masked_pixels=int(mask.sum()),page_pixels=h*w,masked_area_fraction=float(mask.mean()),\n                  source_sha256=hashlib.sha256(src.read_bytes()).hexdigest(), question_types=qas[r[\'question_id\']].get(\'question_types\',[]),\n                  localization_provenance=\'historical automatic answer matching; not manually verified\')\n    audit.append(record)\n    for condition,path in [(\'BASELINE_AUGMENTED\',src),(\'ANSWER_MASKED\',dest)]:\n        manifest.append(dict(r,condition=condition,image_path=str(path.resolve())))\nif len(manifest) != 388: raise RuntimeError(f\'Expected 194 eligible pairs; got {len(manifest)//2}; inspect audit\')\nwith (out/\'manifest.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n    writer=csv.DictWriter(f,fieldnames=list(manifest[0]));writer.writeheader();writer.writerows(manifest)\nreport={\'input_documents\':len(rows),\'eligible_documents\':len(manifest)//2,\'planned_inferences\':len(manifest),\n        \'eligible_roles\':dict(Counter(x[\'role\'] for x in audit if x[\'eligible\'])),\n        \'documents\':audit,\'scope\':\'Oracle answer-region image ablation on augmented inputs; no policy comparison, PII-category inference, or exhaustive answer-occurrence annotation.\'}\n(out/\'annotation_audit.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\nprint(json.dumps({k:v for k,v in report.items() if k!=\'documents\'},indent=2))\n', 'analyze_ablation.py': "import csv,json,sys,re\nfrom pathlib import Path\nimport numpy as np\n\ndef norm(s): return re.sub(r'\\s+',' ',str(s).lower().strip())\ndef edit(a,b):\n    prev=list(range(len(b)+1))\n    for i,ca in enumerate(a,1):\n        cur=[i]\n        for j,cb in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))\n        prev=cur\n    return prev[-1]\ndef score(pred,answers):\n    p=norm(pred); vals=[]\n    for answer in answers:\n        a=norm(answer); distance=edit(p,a)/max(len(p),len(a),1)\n        vals.append(1-distance if distance<.5 else 0.)\n    return max(vals,default=0.)\ndef ci(x):\n    x=np.asarray(x,float);rng=np.random.default_rng(20261003)\n    return np.quantile(rng.choice(x,size=(10000,len(x)),replace=True).mean(axis=1),[.025,.975]).tolist()\npred,out,audit_path=map(Path,sys.argv[1:4]);audit=json.loads(audit_path.read_text());eligible={r['document_id']:r for r in audit['documents'] if r['eligible']}\nlatest={}\nfor line in pred.read_text().splitlines():\n    if not line.strip():continue\n    r=json.loads(line)\n    if r['status']=='ok' and r['document_id'] in eligible:latest[(r['document_id'],r['condition'])]=r\nexpected={(d,c) for d in eligible for c in ['BASELINE_AUGMENTED','ANSWER_MASKED']}\nif set(latest)!=expected:raise RuntimeError(f'Missing {len(expected-set(latest))} successful predictions; rerun inference cell')\nmetrics=[];pairs=[]\nfor d,a in sorted(eligible.items()):\n    pair={'source_document_id':a['source_document_id'],'document_id':d,'role':a['role'],'question_types':'|'.join(a['question_types'])}\n    for c in ['BASELINE_AUGMENTED','ANSWER_MASKED']:\n        r=latest[(d,c)];s=score(r['prediction'],r['answers']);em=float(norm(r['prediction']) in {norm(x) for x in r['answers']})\n        pair[c]=s;pair[c+'_EM']=em\n        metrics.append(dict(document_id=d,condition=c,role=a['role'],question_types=pair['question_types'],prediction=r['prediction'],ANLS=s,exact_match=em))\n    pairs.append(pair)\nsummary=[]\ngroups=[('ALL',pairs)]+[(f'ROLE:{role}',[p for p in pairs if p['role']==role]) for role in sorted({p['role'] for p in pairs})]\nfor t in sorted({t for a in eligible.values() for t in a['question_types']}):groups.append((f'QUESTION_TYPE:{t}',[p for p in pairs if t in p['question_types'].split('|')]))\nfor group,ps in groups:\n    delta=[p['ANSWER_MASKED']-p['BASELINE_AUGMENTED'] for p in ps];lo,hi=ci(delta)\n    clusters=sorted({p['source_document_id'] for p in ps}); sums=np.array([sum(p['ANSWER_MASKED']-p['BASELINE_AUGMENTED'] for p in ps if p['source_document_id']==k) for k in clusters]); counts=np.array([sum(p['source_document_id']==k for p in ps) for k in clusters]); rng=np.random.default_rng(20261003); ix=rng.integers(0,len(clusters),size=(10000,len(clusters))); clo,chi=np.quantile(sums[ix].sum(axis=1)/counts[ix].sum(axis=1),[.025,.975])\n    summary.append(dict(group=group,n=len(ps),source_documents=len(clusters),cluster_CI95_low=float(clo),cluster_CI95_high=float(chi),baseline_ANLS=float(np.mean([p['BASELINE_AUGMENTED'] for p in ps])),masked_ANLS=float(np.mean([p['ANSWER_MASKED'] for p in ps])),difference=float(np.mean(delta)),CI95_low=lo,CI95_high=hi,baseline_EM=float(np.mean([p['BASELINE_AUGMENTED_EM'] for p in ps])),masked_EM=float(np.mean([p['ANSWER_MASKED_EM'] for p in ps])),better=int(np.sum(np.array(delta)>0)),same=int(np.sum(np.array(delta)==0)),worse=int(np.sum(np.array(delta)<0))))\nfor name,data in [('prediction_metrics.csv',metrics),('paired_scores.csv',pairs),('ablation_summary.csv',summary)]:\n    with (out/name).open('w',newline='',encoding='utf-8') as f:w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)\nprint(json.dumps(summary,indent=2))\n", 'run_ablation.mjs': 'import fs from "node:fs";\nimport { env, pipeline } from "@xenova/transformers";\n\nconst manifestPath = process.argv[2];\nconst outputPath = process.argv[3];\nconst cacheDir = process.argv[4];\nconst revision = process.argv[5];\nif (!revision) throw new Error("Model revision is required");\nenv.cacheDir = cacheDir;\nenv.allowLocalModels = false;\nenv.allowRemoteModels = true;\n\nfunction parseCSV(text) {\n  const rows = []; let row = [], field = "", quoted = false;\n  for (let i = 0; i < text.length; i++) {\n    const ch = text[i];\n    if (quoted) {\n      if (ch === \'"\' && text[i + 1] === \'"\') { field += \'"\'; i++; }\n      else if (ch === \'"\') quoted = false; else field += ch;\n    } else if (ch === \'"\') quoted = true;\n    else if (ch === ",") { row.push(field); field = ""; }\n    else if (ch === "\\n") { row.push(field.replace(/\\r$/, "")); rows.push(row); row = []; field = ""; }\n    else field += ch;\n  }\n  if (field.length || row.length) { row.push(field.replace(/\\r$/, "")); rows.push(row); }\n  const header = rows[0];\n  return rows.slice(1).filter(r => r.length === header.length).map(r =>\n    Object.fromEntries(header.map((h, i) => [h, r[i]]))\n  );\n}\n\nconst rows = parseCSV(fs.readFileSync(manifestPath, "utf8"));\nconst validKeys = new Set(rows.map(r => `${r.document_id}\\t${r.condition}`));\nconst completed = new Set();\nif (fs.existsSync(outputPath)) {\n  for (const line of fs.readFileSync(outputPath, "utf8").split(/\\n/)) {\n    if (!line.trim()) continue;\n    try {\n      const x = JSON.parse(line); const key = `${x.document_id}\\t${x.condition}`;\n      if (x.status === "ok" && validKeys.has(key)) completed.add(key);\n    } catch {}\n  }\n}\nconsole.log(`Checkpoint: ${completed.size}/${rows.length}`);\n\nconst pipe = await pipeline(\n  "document-question-answering", "Xenova/donut-base-finetuned-docvqa",\n  { quantized: true, revision }\n);\npipe.processor.feature_extractor.size = { height: 1280, width: 960 };\n\nlet newlyCompleted = 0;\nfor (const row of rows) {\n  const key = `${row.document_id}\\t${row.condition}`;\n  if (completed.has(key)) continue;\n  const start = Date.now(); let prediction = "", status = "ok", error = null;\n  try {\n    const result = await pipe(row.image_path, row.question, { max_new_tokens: 64 });\n    prediction = result?.[0]?.answer ?? "";\n  } catch (e) { status = "error"; error = String(e?.stack ?? e); }\n  const record = {\n    sample_index: Number(row.sample_index), document_id: row.document_id,\n    role: row.role, question_id: Number(row.question_id), question: row.question,\n    answers: JSON.parse(row.answers_json), condition: row.condition, prediction,\n    status, error, inference_seconds: (Date.now() - start) / 1000,\n    model: "Xenova/donut-base-finetuned-docvqa", quantized: true,\n    processor_size: [1280, 960], revision,\n  };\n  fs.appendFileSync(outputPath, JSON.stringify(record) + "\\n");\n  newlyCompleted++;\n  if (newlyCompleted % 25 === 0) console.log(`New: ${newlyCompleted}; total target: ${rows.length}`);\n}\nconsole.log("Inference loop completed");\n\n'}
for name, text in SCRIPTS.items(): (WORK/name).write_text(text,encoding='utf-8')
subprocess.run([sys.executable,str(WORK/'prepare_ablation.py'),str(WORK),str(WORK/'prepared')],check=True)


In [ ]:
import numpy as np, PIL
SPEC_PATH=OUT/'run_spec.json'
MODEL='Xenova/donut-base-finetuned-docvqa'
if SPEC_PATH.exists():
    REVISION=json.loads(SPEC_PATH.read_text())['model_revision']
else:
    with urllib.request.urlopen('https://huggingface.co/api/models/'+MODEL,timeout=60) as response:
        REVISION=json.load(response)['sha']
spec={'experiment':'DocVQA-answer-ablation-v1','model':MODEL,'model_revision':REVISION,
      'input_sha256':sha(INPUT_ZIP),'manifest_sha256':sha(WORK/'prepared/manifest.csv'),
      'script_sha256':{n:sha(WORK/n) for n in SCRIPTS},
      'transformers_js':'2.17.2','node':subprocess.check_output(['node','--version'],text=True).strip(),
      'python':platform.python_version(),'numpy':np.__version__,'pillow':PIL.__version__,
      'quantized':True,'processor_size':[1280,960],'max_new_tokens':64,'bootstrap_seed':20261003,'bootstrap_replicates':10000,
      'scoring':'lowercase/whitespace normalized, punctuation retained, normalized edit distance < 0.5',
      'conditions':['BASELINE_AUGMENTED','ANSWER_MASKED']}
if SPEC_PATH.exists() and json.loads(SPEC_PATH.read_text())!=spec:
    raise RuntimeError('실행 설정이 기존 결과와 다릅니다. 기존 결과 폴더를 백업하고 새 폴더에서 실행하세요.')
SPEC_PATH.write_text(json.dumps(spec,indent=2),encoding='utf-8')
(OUT/'annotation_audit.json').write_bytes((WORK/'prepared/annotation_audit.json').read_bytes())
print('모델 버전:',REVISION)


In [ ]:
# 중단되면 이 셀을 다시 실행합니다. 성공한 추론은 건너뜁니다.
subprocess.run(['node',str(WORK/'run_ablation.mjs'),str(WORK/'prepared/manifest.csv'),
                str(OUT/'predictions.jsonl'),str(CACHE),REVISION],cwd=WORK,check=True)


In [ ]:
subprocess.run([sys.executable,str(WORK/'analyze_ablation.py'),str(OUT/'predictions.jsonl'),
                str(OUT),str(OUT/'annotation_audit.json')],check=True)
import pandas as pd
summary=pd.read_csv(OUT/'ablation_summary.csv')
display(summary)
RESULT=Path('/content/DocVQA_TaskAblation_Results.zip')
with zipfile.ZipFile(RESULT,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['ablation_summary.csv','prediction_metrics.csv','paired_scores.csv','predictions.jsonl','annotation_audit.json','run_spec.json']:
        z.write(OUT/name,arcname=name)
from google.colab import files
files.download(str(RESULT))
print('완료: DocVQA_TaskAblation_Results.zip을 대화에 첨부하세요.')
